In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

WORKSPACE = Path('/kaggle/working')
SOFTGROUP_REPO_URL = 'https://github.com/thangvubk/SoftGroup.git'
SOFTGROUP_DIR = WORKSPACE / 'SoftGroup'
S3DIS_INPUT_DIR = Path('/kaggle/input/datasets/ayukiseleva/s3dis-meow/Stanford3dDataset_v1.2')
S3DIS_REPO_DIR = SOFTGROUP_DIR / 'dataset' / 's3dis'
S3DIS_REPO_RAW_DIR = S3DIS_REPO_DIR / 'Stanford3dDataset_v1.2'
OUTPUT_ZIP = WORKSPACE / 's3dis-preprocessed.zip'

RUN_CROP = False
KEEP_ROOMS = ['office_1', 'office_2', 'office_3']
FORCE_REBUILD_PREPROCESS = True

def run(cmd, cwd=None, env=None, check=True):
    print('\n[RUN]', ' '.join(map(str, cmd)) if isinstance(cmd, (list, tuple)) else cmd, '\n')
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, shell=isinstance(cmd, str), check=check)

def safe_remove(path: Path):
    if not path.exists() and not path.is_symlink():
        return
    if path.is_symlink() or path.is_file():
        path.unlink()
    else:
        shutil.rmtree(path)

assert S3DIS_INPUT_DIR.exists(), f'Missing S3DIS input: {S3DIS_INPUT_DIR}'

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'wheel', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2', 'pandas', 'pyyaml', 'tqdm', 'plyfile'])

if not SOFTGROUP_DIR.exists():
    run(['git', 'clone', SOFTGROUP_REPO_URL, str(SOFTGROUP_DIR)])
else:
    print('SoftGroup already exists:', SOFTGROUP_DIR)

S3DIS_REPO_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def link_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        print('Already exists:', dst)
        return
    try:
        dst.symlink_to(src, target_is_directory=True)
        print('Symlink:', dst, '->', src)
    except OSError as exc:
        print('Symlink failed, copying:', exc)
        shutil.copytree(src, dst)

def copy_crop(src_root: Path, dst_root: Path, keep_rooms):
    safe_remove(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)
    copied = []
    for area_dir in sorted(p for p in src_root.iterdir() if p.is_dir() and p.name.startswith('Area_')):
        dst_area = dst_root / area_dir.name
        dst_area.mkdir(parents=True, exist_ok=True)
        for room in keep_rooms:
            src_room = area_dir / room
            if src_room.is_dir():
                shutil.copytree(src_room, dst_area / room)
                copied.append(f'{area_dir.name}/{room}')
    print('Copied rooms:', len(copied))

if RUN_CROP:
    copy_crop(S3DIS_INPUT_DIR, S3DIS_REPO_RAW_DIR, KEEP_ROOMS)
else:
    safe_remove(S3DIS_REPO_RAW_DIR)
    link_or_copy(S3DIS_INPUT_DIR, S3DIS_REPO_RAW_DIR)

print('Raw S3DIS ready:', S3DIS_REPO_RAW_DIR)

In [ ]:
prepare_inst_path = S3DIS_REPO_DIR / 'prepare_data_inst.py'
src = prepare_inst_path.read_text(encoding='utf-8')
patched = src
patched = re.sub(
    r"room_ver\s*=\s*pd\.read_csv\(\s*raw_path\s*,\s*sep=['\"] ['\"]\s*,\s*header=None\s*\)\.values",
    "room_ver = pd.read_csv(raw_path, sep=r'\\s+', header=None, engine='python').apply(pd.to_numeric, errors='coerce').dropna(axis=0, how='any').values",
    patched,
)
patched = re.sub(
    r"obj_ver\s*=\s*pd\.read_csv\(\s*single_object\s*,\s*sep=['\"] ['\"]\s*,\s*header=None\s*\)\.values",
    "obj_ver = pd.read_csv(single_object, sep=r'\\s+', header=None, engine='python').apply(pd.to_numeric, errors='coerce').dropna(axis=0, how='any').values",
    patched,
)
patched = patched.replace(
    "rgb = np.ascontiguousarray(room_ver[:, 3:6], dtype='uint8')",
    "rgb = np.clip(np.rint(np.ascontiguousarray(room_ver[:, 3:6], dtype='float32')), 0, 255).astype('uint8')",
)
if patched != src:
    prepare_inst_path.write_text(patched, encoding='utf-8')
    print('Patched:', prepare_inst_path)
else:
    print('Parser patch already applied or not needed.')

In [ ]:
if FORCE_REBUILD_PREPROCESS:
    for name in ['preprocess', 'preprocess_sample', 'val_gt']:
        safe_remove(S3DIS_REPO_DIR / name)

run(['bash', 'prepare_data.sh'], cwd=S3DIS_REPO_DIR)

processed_files = sorted((S3DIS_REPO_DIR / 'preprocess').glob('*_inst_nostuff.pth'))
print('Processed room files:', len(processed_files))
assert processed_files, 'prepare_data.sh finished but preprocess/*.pth is empty'

In [ ]:
safe_remove(OUTPUT_ZIP)
metadata = {
    'source': str(S3DIS_INPUT_DIR),
    'softgroup_repo': SOFTGROUP_REPO_URL,
    'rooms': len(list((S3DIS_REPO_DIR / 'preprocess').glob('*_inst_nostuff.pth'))),
    'contains': ['preprocess', 'preprocess_sample', 'val_gt'],
}

with zipfile.ZipFile(OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=4) as zf:
    zf.writestr('preprocess_metadata.json', json.dumps(metadata, indent=2))
    for folder_name in ['preprocess', 'preprocess_sample', 'val_gt']:
        folder = S3DIS_REPO_DIR / folder_name
        if not folder.exists():
            continue
        for path in folder.rglob('*'):
            if path.is_file():
                zf.write(path, arcname=str(path.relative_to(S3DIS_REPO_DIR)))

print('Created:', OUTPUT_ZIP)
print('Size GB:', OUTPUT_ZIP.stat().st_size / 1024**3)